# YouTube Playlist Sorter with Gemini AI

This notebook automates the process of sorting your YouTube "Watch Later" (or any other) playlist into neatly categorized new playlists using the Google Gemini 1.5 AI model.

### Workflow:
1.  **Configuration**: Set your parameters in the second cell, especially `DRY_RUN` mode for safety.
2.  **Authentication**: The main script will guide you through authenticating with your YouTube account and Gemini API.
3.  **Execution**: The script fetches video metadata, uses Gemini's powerful File API to classify all videos in a single, efficient call, and then (if not in `DRY_RUN` mode) creates new private playlists for each category.
4.  **Reporting**: At the end, it generates and downloads CSV and Excel reports of the classification.

### 🔑 Prerequisites: Colab Secrets

Before running, you must add the following **secrets** to your Colab environment (click the **key icon (🔑)** in the left-hand panel):

- **`GEMINI_API_KEY`**: Your API key from [Google AI Studio](https://aistudio.google.com/app/apikey).
- **`YT_CLIENT_ID`**: Your OAuth 2.0 Client ID from the Google Cloud Console. **Important**: It must be for a **Desktop App**.
- **`YT_CLIENT_SECRET`**: Your OAuth 2.0 Client Secret associated with the Client ID.

## Étape 1 : Paramètres Globaux

Configurez le comportement du script ici. Le paramètre le plus important est `DRY_RUN`.

- **`DRY_RUN = True`**: (Mode sans risque) Le script effectue toutes les analyses et génère des rapports, mais **ne créera aucune playlist** et ne modifiera pas votre compte YouTube. **Recommandé pour la première exécution.**
- **`DRY_RUN = False`**: (Mode Live) Le script **créera** de nouvelles playlists privées sur votre compte YouTube.

In [6]:
# ==============================================================================
#   ÉTAPE 1 : PARAMÈTRES GLOBAUX (corrigés)
# ==============================================================================
SOURCE_MODE = "csv"   # 'api' ou 'csv'
MAX_VIDEOS = 5010
DRY_RUN = False
PLAYLIST_PREFIX = "PG"

# Modèle : choisissez un alias garanti par le SDK en place
LLM_MODEL = "gemini-2.5-flash-lite"

YOUTUBE_BATCH_SIZE = 50

SAVE_REPORT_CSV = "rapport_de_tri.csv"
SAVE_REPORT_XLSX = "rapport_de_tri.xlsx"


## Étape 2 : Installation et Définition des Fonctions

Cette cellule installe toutes les librairies nécessaires et définit toutes les fonctions qui seront utilisées dans les étapes suivantes. Vous n'avez besoin d'exécuter cette cellule qu'une seule fois par session.

In [1]:
# ==============================================================================
#   ÉTAPE 2 : INSTALL + IMPORTS (stables)
# ==============================================================================
!pip -q install --upgrade google-generativeai google-api-python-client google-auth-httplib2 google-auth-oauthlib pandas tqdm openpyxl ipywidgets

import os, csv, json, random, re, time
from typing import Dict, List

import google.generativeai as genai
import pandas as pd
from google.colab import files, userdata
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm.auto import tqdm
from urllib.parse import parse_qs, urlparse

print("✅ Librairies importées avec succès.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 whi

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Librairies importées avec succès.


In [2]:
# ==============================================================================
#   AUTH + YT HELPERS (ajoutés)
# ==============================================================================
def setup_apis_and_auth_secrets():
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

    gemini_key = (userdata.get('GEMINI_API_KEY') or "").strip()
    if not gemini_key:
        raise ValueError("Le secret 'GEMINI_API_KEY' est manquant.")
    genai.configure(api_key=gemini_key)
    llm = genai.GenerativeModel(LLM_MODEL)
    print(f"✅ API Gemini prête (modèle : {LLM_MODEL}).")

    client_id = (userdata.get('YT_CLIENT_ID') or "").strip()
    client_secret = (userdata.get('YT_CLIENT_SECRET') or "").strip()
    if not client_id or not client_secret:
        raise ValueError("Les secrets 'YT_CLIENT_ID' et/ou 'YT_CLIENT_SECRET' sont manquants.")

    client_config = {
        "installed": {
            "client_id": client_id, "client_secret": client_secret,
            "auth_uri": "https://accounts.google.com/o/oauth2/auth",
            "token_uri": "https://oauth2.googleapis.com/token",
            "redirect_uris": ["http://localhost"]
        }
    }
    scopes = ["https://www.googleapis.com/auth/youtube.readonly"]
    if not DRY_RUN:
        scopes.append("https://www.googleapis.com/auth/youtube")

    flow = Flow.from_client_config(client_config, scopes=scopes, redirect_uri="http://localhost")
    auth_url, _ = flow.authorization_url(
        access_type="offline", include_granted_scopes="true", prompt="consent"
    )
    print("\n--- Authentification YouTube requise ---")
    print(f"1. Ouvrez cette URL:\n   🔗 {auth_url}")
    print("\n2. Collez l'URL ENTIER ou le CODE d'autorisation ici, puis Entrée:")

    raw_input_str = input("   ").strip()
    code = parse_qs(urlparse(raw_input_str).query).get("code", [""])[0] if raw_input_str.startswith("http") else raw_input_str
    if not code:
        raise ValueError("Code d'autorisation non trouvé. Reprenez la procédure et collez bien l'URL ou le code.")
    flow.fetch_token(code=code)

    youtube_service = build("youtube", "v3", credentials=flow.credentials)
    print(f"✅ API YouTube authentifiée avec succès.")
    return youtube_service, llm

def get_watch_later_playlist_id(youtube):
    # Watch Later est une playlist spéciale "WL"
    # Lister pour être robuste si WL change (lecture seule)
    resp = youtube.channels().list(part="contentDetails", mine=True).execute()
    items = resp.get("items", [])
    if not items:
        raise RuntimeError("Impossible de récupérer le channel courant.")
    wl_id = items[0]["contentDetails"]["relatedPlaylists"]["watchLater"]
    return wl_id

def fetch_playlist_video_ids(youtube, playlist_id: str, max_items: int = 5000) -> List[str]:
    ids = []
    page_token = None
    with tqdm(desc="Récupération des IDs", unit="vidéo") as pbar:
        while True:
            req = youtube.playlistItems().list(
                part="contentDetails",
                playlistId=playlist_id,
                maxResults=50,
                pageToken=page_token
            )
            resp = req.execute()
            for it in resp.get("items", []):
                vid = it["contentDetails"]["videoId"]
                ids.append(vid)
            pbar.update(len(resp.get("items", [])))
            page_token = resp.get("nextPageToken")
            if not page_token or len(ids) >= max_items:
                break
    return ids[:max_items]


##BREAK HERE##


In [ ]:
# ==============================================================================
#   CAT SUGGESTION (sans outils, JSON strict)
# ==============================================================================
def propose_15_categories(llm, video_items: List[Dict], max_retries: int = 3) -> List[str]:
    print("🧠 Analyse du corpus par Gemini (catégories)...")
    sample_size = min(500, len(video_items))
    sample_items = random.sample(video_items, sample_size) if video_items else []
    sample_data = [
        {
            "title": item.get('snippet', {}).get('title', ''),
            "channel": item.get('snippet', {}).get('channelTitle', '')
        } for item in sample_items
    ]
    payload = json.dumps(sample_data, ensure_ascii=False, indent=2)

    prompt = (

    "You are a YouTube taxonomy expert.\n"
    "Analyze the attached JSON sample of videos (title and channel).\n\n"
    "Your goal: produce EXACTLY 15 category names in English.\n"
    "Rules:\n"
    "1. Categories MUST be mutually exclusive (no overlaps in meaning).\n"
    "2. If two topics are similar (e.g., 'Personal Development' and 'Self-Improvement'), "
    "merge them into ONE, choosing the clearest, most widely used term.\n"
    "3. Use short, descriptive names (2–4 words).\n"
    "4. Avoid overly generic or catch-all names (e.g., 'Technology', 'Miscellaneous', 'General').\n"
    "5. Ensure coverage across diverse domains: science, technology, business, health, hobbies, art, history, etc.\n"
    "6. Do not repeat the same theme with synonyms.\n"
    "7. Order categories from most to least frequent in the dataset."
    "8. Suggested categories are  : 'Psychology & Self-Improvement','AI, Coding & Systems','Strength, Fitness & Longevity', 'Philosophy & Big Ideas','History & Civilization','Mathematics','UFOs, Anomalies & Esoterica', 'Business & Finance','Cooking & Food', 'BJJ, Grappling & Combat Sports','Chess & Strategy', 'Science & Reality', 'Art, Design & Creativity', 'Language & Communication'\n\n"
    "Return ONLY a JSON array of strings, nothing else."
    "SAMPLE_JSON:\n```json\n" + payload + "\n```"
    )

    for attempt in range(1, max_retries+1):
        try:
            # ✅ Send everything as TEXT (no application/json part)
            resp = llm.generate_content(
                [{"text": prompt}],
                request_options={"timeout": 120}
            )
            text = (resp.text or "").strip()
            m = re.search(r'\[.*\]', text, flags=re.S)
            arr = json.loads(m.group(0)) if m else json.loads(text)
            arr = [str(x).strip() for x in arr if str(x).strip()]
            arr = list(dict.fromkeys(arr))
            if len(arr) >= 15:
                return arr[:15]
        except Exception as e:
            print(f"   -> ⚠️ Tentative {attempt}/{max_retries} : {e}")
            time.sleep(2**attempt)

    print("   -> 🛑 Impossible d'obtenir 15 catégories, fallback appliqué.")
    return ["General"] * 15


In [ ]:
def get_video_details(youtube_service, video_ids: List[str]) -> List[Dict]:
    items = []
    batches = [video_ids[i:i + YOUTUBE_BATCH_SIZE] for i in range(0, len(video_ids), YOUTUBE_BATCH_SIZE)]
    for batch in tqdm(batches, desc="Métadonnées YouTube", unit="lot"):
        for attempt in range(5):
            try:
                resp = youtube_service.videos().list(part="snippet", id=",".join(batch)).execute()
                items.extend(resp.get("items", []))
                break
            except HttpError as e:
                if getattr(e, "resp", None) and e.resp.status in [403, 429, 500, 503] and attempt < 4:
                    sleep_time = (2 ** attempt) + random.uniform(0, 1)
                    tqdm.write(f"⚠️ Quota/Erreur transitoire. Nouvelle tentative dans {sleep_time:.1f}s...")
                    time.sleep(sleep_time)
                else:
                    tqdm.write(f"🛑 Erreur API non récupérable: {e}")
                    break
    return items

def read_watch_later_csv(uploaded_filename: str) -> List[str]:
    with open(uploaded_filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader, None)
        return [row[0].strip() for row in reader if row and row[0]]

def build_records_from_video_items(video_items: List[dict]) -> List[dict]:
    recs = []
    for item in video_items:
        snip = item.get("snippet", {})
        recs.append({
            "video_id": item.get("id"),
            "video_title": snip.get("title", ""),
            "channel": snip.get("channelTitle", ""),
            "publishedAt": snip.get("publishedAt", None),
        })
    return recs


In [ ]:
def find_json_block(text: str) -> str:
    # Return the first JSON array/object found
    m = re.search(r'```json\s*(\{.*?\}|\[.*?\])\s*```', text, re.S)
    if m: return m.group(1)
    m = re.search(r'(\{.*?\}|\[.*?\])', text, re.S)
    if m: return m.group(1)
    raise ValueError("Aucun bloc JSON détecté dans la réponse LLM.")

def normalize_with_synonyms(raw_cat: str, synonyms_map: Dict[str, str], categories: List[str]) -> str:
    if not raw_cat: return categories[0]
    raw_lower = raw_cat.strip().lower()
    if raw_lower in synonyms_map: return synonyms_map[raw_lower]
    for cat in categories:
        if raw_lower == cat.lower(): return cat
    for cat in categories:
        if cat.lower() in raw_lower or raw_lower in cat.lower(): return cat
    return categories[0]

def _classify_chunk_with_file_api(llm, chunk_df: pd.DataFrame, categories: List[str], pbar, chunk_num, total_chunks) -> Dict[str, str]:
    temp_csv_path = "temp_chunk_data.csv"
    chunk_df.to_csv(temp_csv_path, index=False, encoding="utf-8-sig")
    uploaded_file = genai.upload_file(path=temp_csv_path, display_name="Chunk for classification")
    prompt = (
        "Tu es un classificateur expert de vidéos YouTube.\n"
        "Traite le CSV joint avec colonnes: video_id, video_title, channel.\n"
        f"Catégories AUTORISÉES (choisir EXACTEMENT UNE):\n{json.dumps(categories, ensure_ascii=False)}\n\n"
        "Réponds UNIQUEMENT par un JSON array, format:\n"
        '[{"video_id":"id1","assigned_category":"<cat>"}, ...]'
    )
    try:
        pbar.write(f"  [Lot {chunk_num}/{total_chunks}] 🧠 Analyse en cours...")
        resp = llm.generate_content([prompt, uploaded_file], request_options={"timeout": 600})
        parsed = json.loads(find_json_block(resp.text))
        return {row["video_id"]: row["assigned_category"] for row in parsed if "video_id" in row and "assigned_category" in row}
    finally:
        try:
            genai.delete_file(uploaded_file.name)
        except Exception:
            pass
        try:
            os.remove(temp_csv_path)
        except Exception:
            pass

def classify_dataframe_in_chunks(llm, df: pd.DataFrame, categories: List[str], synonyms_map: Dict[str, str], chunk_size: int = 250, max_retries: int = 5) -> Dict[str, str]:
    final_results = {}
    total = len(df)
    num_chunks = (total + chunk_size - 1) // chunk_size
    with tqdm(total=total, desc="Classification Globale", unit="vidéo") as pbar:
        for i in range(0, total, chunk_size):
            current_chunk_num = i // chunk_size + 1
            chunk_df = df.iloc[i:i + chunk_size]
            for attempt in range(max_retries):
                try:
                    pbar.write(f"  [Lot {current_chunk_num}/{num_chunks}] 📤 Envoi de {len(chunk_df)} vidéos (tentative {attempt+1}/{max_retries})...")
                    chunk_result = _classify_chunk_with_file_api(llm, chunk_df, categories, pbar, current_chunk_num, num_chunks)  # ✅ pass num_chunks
                    normalized = {vid: normalize_with_synonyms(cat, synonyms_map, categories) for vid, cat in chunk_result.items()}
                    final_results.update(normalized)
                    pbar.write(f"  [Lot {current_chunk_num}/{num_chunks}] ✅ OK.")
                    break
                except Exception as e:
                    pbar.write(f"  [Lot {current_chunk_num}/{num_chunks}] 🛑 ERREUR: {type(e).__name__}: {e}")
                    if attempt < max_retries - 1:
                        sleep_time = 2 ** (attempt + 1)
                        pbar.write(f"      -> Nouvelle tentative dans {sleep_time}s...")
                        time.sleep(sleep_time)
                    else:
                        pbar.write("      -> Échec final pour ce lot, il sera ignoré.")
            pbar.update(len(chunk_df))
    return final_results


##AUTH##

In [8]:
# ==============================================================================
#   ÉTAPE 3 : AUTH
# ==============================================================================
try:
    youtube, llm_model = setup_apis_and_auth_secrets()
except Exception as e:
    print(f"🛑 Une erreur est survenue lors de l'authentification : {e}")


✅ API Gemini prête (modèle : gemini-2.5-flash-lite).

--- Authentification YouTube requise ---
1. Ouvrez cette URL:
   🔗 https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=728085756312-g9r90gdto7vjidi994p09oricp71lf0s.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube&state=Gxfeqo6YkknSW5r9MU2wwNFnpSYYaG&code_challenge=kf93BCsru7_g14qLC9_pLMwMs7FJlFiL8o09Gdhxaow&code_challenge_method=S256&access_type=offline&include_granted_scopes=true&prompt=consent

2. Collez l'URL ENTIER ou le CODE d'autorisation ici, puis Entrée:
   http://localhost/?state=Gxfeqo6YkknSW5r9MU2wwNFnpSYYaG&iss=https://accounts.google.com&code=4/0AfrIepC5ePIiZauJBncem_DMfq8kxohIPSimlkVWteLs1FqyqgHm5wdthmpOyaIZ9NkNjg&scope=https://www.googleapis.com/auth/youtube.readonly%20https://www.googleapis.com/auth/youtube
✅ API YouTube authentifiée avec succès.


##BREAK##

In [ ]:
# ====================== HOTFIX HELPERS (run once) ======================
import csv, json, time, random, re, os
from typing import List, Dict
import pandas as pd
from tqdm.auto import tqdm
from googleapiclient.errors import HttpError

# Read IDs from a CSV exported from Watch Later (first column = videoId)
def read_watch_later_csv(uploaded_filename: str) -> List[str]:
    ids = []
    with open(uploaded_filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader, None)  # skip header if present
        for row in reader:
            if not row:
                continue
            vid = (row[0] or "").strip()
            if vid:
                ids.append(vid)
    return ids

# Optional: fetch full metadata for a list of IDs
def get_video_details(youtube_service, video_ids: List[str], batch_size: int = 50) -> List[Dict]:
    items = []
    batches = [video_ids[i:i+batch_size] for i in range(0, len(video_ids), batch_size)]
    for batch in tqdm(batches, desc="Métadonnées YouTube", unit="lot"):
        for attempt in range(5):
            try:
                resp = youtube_service.videos().list(
                    part="snippet",
                    id=",".join(batch)
                ).execute()
                items.extend(resp.get("items", []))
                break
            except HttpError as e:
                if getattr(e, "resp", None) and e.resp.status in [403, 429, 500, 503] and attempt < 4:
                    wait = (2 ** attempt) + random.uniform(0, 1)
                    tqdm.write(f"⚠️ Quota/Erreur transitoire. Retry dans {wait:.1f}s…")
                    time.sleep(wait)
                else:
                    tqdm.write(f"🛑 Erreur API non récupérable: {e}")
                    break
    return items

# Build a clean records list from videoItems
def build_records_from_video_items(video_items: List[dict]) -> List[dict]:
    recs = []
    for it in video_items:
        sn = it.get("snippet", {})
        recs.append({
            "video_id": it.get("id"),
            "video_title": sn.get("title", ""),
            "channel": sn.get("channelTitle", ""),
            "publishedAt": sn.get("publishedAt", None),
        })
    return recs

# If you plan to use SOURCE_MODE="api", you’ll also need these:
def get_watch_later_playlist_id(youtube):
    resp = youtube.channels().list(part="contentDetails", mine=True).execute()
    items = resp.get("items", [])
    if not items:
        raise RuntimeError("Impossible de récupérer le channel courant.")
    return items[0]["contentDetails"]["relatedPlaylists"]["watchLater"]

def fetch_playlist_video_ids(youtube, playlist_id: str, max_items: int = 5000) -> List[str]:
    ids, token = [], None
    with tqdm(desc="Récupération des IDs", unit="vidéo") as pbar:
        while True:
            req = youtube.playlistItems().list(
                part="contentDetails", playlistId=playlist_id,
                maxResults=50, pageToken=token
            )
            resp = req.execute()
            batch = [it["contentDetails"]["videoId"] for it in resp.get("items", [])]
            ids.extend(batch); pbar.update(len(batch))
            token = resp.get("nextPageToken")
            if not token or len(ids) >= max_items: break
    return ids[:max_items]

print("✅ HOTFIX helpers chargés.")



✅ HOTFIX helpers chargés.


In [ ]:
# ==============================================================================
#   ÉTAPE 4 : CHARGEMENT & MÉTADONNÉES
# ==============================================================================
if 'youtube' in locals():
    if SOURCE_MODE.lower() == "csv":
        print("\n👉 Veuillez importer votre fichier CSV (ex: 'Vidéos de Watch later.csv').")
        uploaded = files.upload()
        if not uploaded:
            raise SystemExit("🛑 ERREUR : Aucun fichier n'a été uploadé.")
        csv_name = next(iter(uploaded.keys()))
        print(f"📄 Fichier sélectionné : {csv_name}")
        video_ids = read_watch_later_csv(csv_name)
    else:
        print("\n🔗 Récupération des vidéos via l'API YouTube (playlist 'À regarder plus tard')...")
        wl_id = get_watch_later_playlist_id(youtube)
        print(f"✅ ID de la playlist 'À regarder plus tard' trouvé : {wl_id}")
        video_ids = fetch_playlist_video_ids(youtube, wl_id, max_items=MAX_VIDEOS)

    video_ids = (video_ids or [])[:MAX_VIDEOS]
    if not video_ids:
        raise SystemExit("🛑 ERREUR : Aucun ID de vidéo n'a été trouvé.")
    print(f"✅ {len(video_ids)} ID(s) de vidéo collectés.")

    print("\n➡️  Récupération des métadonnées (titres, chaînes...) pour chaque vidéo...")
    video_items = get_video_details(youtube, video_ids)
    print(f"✅ {len(video_items)} métadonnées de vidéos récupérées. Prêt pour l'étape 5.")
else:
    print("🛑 Veuillez d'abord exécuter la cellule d'authentification.")



👉 Veuillez importer votre fichier CSV (ex: 'Vidéos de Watch later.csv').


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
#   ÉTAPE 5 : CATÉGORIES + CLASSIFICATION
# ==============================================================================
from google.colab import output
output.enable_custom_widget_manager()

CLASSIFICATION_CHUNK_SIZE = 250

if 'video_items' in locals() and 'llm_model' in locals():
    print("--- Démarrage : Pipeline de Classification Complet ---")

    print("\n➡️  1/4 : Suggestion des catégories...")
    initial_categories = propose_15_categories(llm_model, video_items)
    print("✅ Suggestion des 15 catégories terminée.")
    print("\n--- Catégories Proposées ---")
    for i, cat in enumerate(initial_categories, 1):
        print(f"{i:02d}. {cat}")


In [ ]:
    current_categories = initial_categories.copy()
    synonyms_map = {c.lower(): c for c in current_categories}

    print(f"\n➡️  2/4 : Préparation des {len(video_items)} vidéos pour la classification...")
    records = build_records_from_video_items(video_items)
    df_for_classification = pd.DataFrame(records)
    print("✅ Données prêtes sous forme de DataFrame.")

    print(f"\n➡️  3/4 : Classification (lots de {CLASSIFICATION_CHUNK_SIZE})...")
    id_to_cat_map = classify_dataframe_in_chunks(
        llm_model,
        df_for_classification[['video_id', 'video_title', 'channel']],
        current_categories,
        synonyms_map,
        chunk_size=CLASSIFICATION_CHUNK_SIZE
    )
    print(f"\n✅ Classification terminée. {len(id_to_cat_map)} vidéos ont reçu une catégorie.")

    print("\n➡️  4/4 : DataFrame final + aperçu")
    classified_df = df_for_classification.copy()
    classified_df['assigned_category'] = classified_df['video_id'].map(id_to_cat_map)
    classified_df['assigned_category'] = classified_df['assigned_category'].fillna('Non Catégorisé')

    # publishDates already inside records; ensure proper type
    classified_df['publishedAt'] = pd.to_datetime(classified_df['publishedAt'], errors='coerce')
    recent_videos_df = classified_df.sort_values(by='publishedAt', ascending=False).head(15)

    print("\n--- Titres et Catégories des 15 Vidéos les plus Récentes ---")
    for _, row in recent_videos_df.iterrows():
        print(f'- "{row["video_title"]}"  ->  Catégorie : {row["assigned_category"]}')

    print("\n✅ Étape 5 terminée ! Le DataFrame `classified_df` est prêt pour l'exportation.")



In [ ]:
# ==============================================================================
#   ÉTAPE 6 : EXPORT CSV
# ==============================================================================
if 'classified_df' in locals():
    print("--- Démarrage de l'étape 6 : Exportation des données ---")
    output_filename_csv = 'watch_later_classified.csv'
    df_to_export = classified_df[['video_id', 'video_title', 'channel', 'assigned_category']].copy()
    df_to_export.rename(columns={'assigned_category': 'category'}, inplace=True)
    df_to_export.to_csv(output_filename_csv, index=False, encoding='utf-8-sig')
    print(f"✅ Fichier '{output_filename_csv}' créé avec succès.")
    print("🚀 Téléchargement du fichier...")
    files.download(output_filename_csv)
else:
    print("🛑 Le DataFrame `classified_df` n'a pas été trouvé. Veuillez exécuter l'étape 5.")


In [ ]:
# ==============================================================================
#   ÉTAPE 7 (VERSION QUOTA-AWARE & RESUMABLE)
# ==============================================================================
import json
from math import ceil

# --- CONFIG ---
DAILY_QUOTA_BUDGET = 9000   # keep under 10,000 default limit
RESUME_FILE = "remaining_to_add.json"

if 'classified_df' not in locals():
    print("🛑 Le DataFrame `classified_df` n'a pas été trouvé. Exécutez l'Étape 5 avant.")
else:
    print("--- Démarrage de l'étape 7 (Quota-aware) ---")

    # --- 1. Préparer données ---
    df_report = classified_df.copy()
    df_report['url'] = 'https://www.youtube.com/watch?v=' + df_report['video_id']
    categorized = df_report.groupby('assigned_category')['video_id'].apply(list).to_dict()

    # --- 2. Quota cost estimation ---
    total_playlists = sum(1 for cat, vids in categorized.items() if vids and cat != "Non Catégorisé")
    total_videos = sum(len(vids) for cat, vids in categorized.items() if vids and cat != "Non Catégorisé")
    estimated_units = (total_playlists * 50) + (total_videos * 50)
    print(f"\n📊 Estimation du coût API : {estimated_units} unités (quotas par défaut = 10,000/jour)")

    if estimated_units > DAILY_QUOTA_BUDGET:
        print(f"⚠️ Cela dépasse le budget journalier ({DAILY_QUOTA_BUDGET} unités). "
              f"Le script s'arrêtera automatiquement lorsque la limite sera atteinte.")

    # --- 3. Charger reprise si disponible ---
    remaining_tasks = []
    if os.path.exists(RESUME_FILE):
        print(f"♻️ Fichier de reprise trouvé : {RESUME_FILE}. Chargement…")
        with open(RESUME_FILE, 'r', encoding='utf-8') as f:
            remaining_tasks = json.load(f)
    else:
        for category, vids in categorized.items():
            if vids and category != "Non Catégorisé":
                remaining_tasks.append({
                    "category": category,
                    "videos": vids
                })

    # --- 4. Playlist creation + ajout vidéos ---
    moved_count = 0
    used_units = 0
    still_remaining = []

    if DRY_RUN:
        print("\n🧪 Mode dry-run actif. Aucun changement sur YouTube.")
    else:
        for task in tqdm(remaining_tasks, desc="Traitement des catégories"):
            category = task["category"]
            vids = task["videos"]

            # Vérifier quota avant de créer playlist
            if used_units + 50 > DAILY_QUOTA_BUDGET:
                still_remaining.append(task)
                continue

            # Création playlist
            try:
                pl_resp = youtube.playlists().insert(
                    part="snippet,status",
                    body={
                        "snippet": {
                            "title": f"{PLAYLIST_PREFIX} {category}",
                            "description": f"Playlist générée automatiquement pour '{category}'."
                        },
                        "status": {"privacyStatus": "private"}
                    }
                ).execute()
                pl_id = pl_resp['id']
                used_units += 50
                tqdm.write(f"📁 Playlist créée : '{category}' (50 unités)")
            except HttpError as e:
                tqdm.write(f"🛑 Erreur création playlist '{category}': {e}")
                continue

            # Ajout vidéos
            remaining_vids = []
            for video_id in vids:
                if used_units + 50 > DAILY_QUOTA_BUDGET:
                    remaining_vids.append(video_id)
                    continue
                try:
                    youtube.playlistItems().insert(
                        part="snippet",
                        body={
                            "snippet": {
                                "playlistId": pl_id,
                                "resourceId": {"kind": "youtube#video", "videoId": video_id}
                            }
                        }
                    ).execute()
                    moved_count += 1
                    used_units += 50
                    time.sleep(0.1)
                except HttpError as e:
                    tqdm.write(f"  -> Erreur ajout {video_id}: {e}")
            if remaining_vids:
                still_remaining.append({"category": category, "videos": remaining_vids})

    # --- 5. Sauvegarde de reprise ---
    if still_remaining:
        with open(RESUME_FILE, 'w', encoding='utf-8') as f:
            json.dump(still_remaining, f, indent=2, ensure_ascii=False)
        print(f"\n⏸ Exécution arrêtée pour respecter la limite. "
              f"{len(still_remaining)} catégories restent à traiter.")
        print(f"💾 Sauvegardé dans {RESUME_FILE}. Relancez demain pour continuer.")
    else:
        if os.path.exists(RESUME_FILE):
            os.remove(RESUME_FILE)
        print("\n✅ Toutes les vidéos ont été déplacées.")

    print(f"\nRésumé : {moved_count} vidéos déplacées aujourd'hui, {used_units} unités utilisées.")



##GOOD##

In [10]:
# Auto-mount Google Drive (runs once per fresh runtime)
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)  # set True if you want to remount every time
    DRIVE_MOUNTED = True
    print("✅ Google Drive mounted at /content/drive")
except Exception as e:
    print("⚠️ Google Drive not available, falling back to local storage:", e)


Mounted at /content/drive
✅ Google Drive mounted at /content/drive


In [11]:
# ==============================================================================
#   ÉTAPE 7B (AUTONOME + GOOGLE DRIVE)
#   • Reuse/auto-recreate playlists sans duplicats
#   • Skip des vidéos déjà présentes dans la playlist
#   • Quota-aware + reprise multi-jours
#   • Monte Google Drive, lit/écrit l'état dans /MyDrive
#   • Ne dépend QUE de l’auth (Étape 3) — pas de classified_df requis
# ==============================================================================

import os, sys, json, re, time, datetime, shutil
import pandas as pd
from tqdm.auto import tqdm
from googleapiclient.errors import HttpError

# ---------------- CONFIG GÉNÉRALE ----------------
DAILY_QUOTA_BUDGET = 9000                        # Budget/jour (défaut YouTube = 10k)
PLAYLIST_PREFIX    = globals().get("PLAYLIST_PREFIX", "YTPG")
DRY_RUN            = globals().get("DRY_RUN", True)
INCLUDE_UNCATEGORIZED = False                    # True => inclure “Non Catégorisé”

# Fichiers (noms) d'état
PLAYLIST_MAP_NAME  = "playlist_map.json"         # mapping {category: playlistId}
RESUME_NAME        = "remaining_to_add.json"     # backlog [{category, videos:[...]}, ...]
REPORT_CSV_NAME    = globals().get("SAVE_REPORT_CSV", "rapport_de_tri.csv")
REPORT_XLSX_NAME   = globals().get("SAVE_REPORT_XLSX", "rapport_de_tri.xlsx")

# ---------------- GOOGLE DRIVE: MONTAGE + RÉPERTOIRE ----------------
USE_DRIVE = True
DRIVE_MOUNTED = False
STATE_DIR = "/content/drive/MyDrive/youtube_sorter_state"  # dossier dédié

def mount_drive_if_needed():
    global DRIVE_MOUNTED
    if not USE_DRIVE:
        return False
    try:
        import google.colab  # type: ignore
        from google.colab import drive  # type: ignore
        if not DRIVE_MOUNTED:
            drive.mount('/content/drive', force_remount=False)
            DRIVE_MOUNTED = True
        return True
    except Exception:
        return False

def ensure_state_dir():
    if USE_DRIVE and DRIVE_MOUNTED:
        os.makedirs(STATE_DIR, exist_ok=True)

def drive_path(name: str) -> str:
    return os.path.join(STATE_DIR, name)

# Écriture atomique + backup horodaté dans Drive
def save_json_atomic_to_drive(name: str, data, make_backup: bool = True):
    path = drive_path(name) if (USE_DRIVE and DRIVE_MOUNTED) else name
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)  # écriture atomique

    if make_backup and USE_DRIVE and DRIVE_MOUNTED:
        ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        shutil.copy2(path, f"{path}.{ts}.bak.json")

def load_json_from_drive_or_local(name: str):
    # Priorité Drive
    if USE_DRIVE and DRIVE_MOUNTED:
        p = drive_path(name)
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
    # Fallback local
    if os.path.exists(name):
        with open(name, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

def file_exists_drive_or_local(name: str) -> bool:
    return (USE_DRIVE and DRIVE_MOUNTED and os.path.exists(drive_path(name))) or os.path.exists(name)

# ---------------- GARDE: API YOUTUBE DISPONIBLE ----------------
if 'youtube' not in globals():
    raise SystemExit("🛑 L'API YouTube n'est pas disponible. Exécutez d'abord l’Étape 3 (auth).")

print("=== Étape 7B (Autonome + Drive) — démarrage ===")
print(f"• DRY_RUN = {DRY_RUN}")
print(f"• Budget quotidien (unités) = {DAILY_QUOTA_BUDGET}")

# Monte Drive + crée le répertoire d'état
if USE_DRIVE:
    if mount_drive_if_needed():
        ensure_state_dir()
        print(f"📁 État persistant → {STATE_DIR}")
    else:
        print("⚠️ Google Drive non disponible — utilisation du stockage local.")

# ---------------- HELPERS: playlists ----------------
def list_user_playlists_by_title(youtube):
    """Retourne {title: id} pour TOUTES les playlists du compte (mine=True)."""
    title_to_id, token = {}, None
    while True:
        resp = youtube.playlists().list(part="snippet", mine=True, maxResults=50, pageToken=token).execute()
        for it in resp.get("items", []):
            title = (it.get("snippet", {}) or {}).get("title", "").strip()
            plid  = it.get("id")
            if title and plid:
                title_to_id[title] = plid
        token = resp.get("nextPageToken")
        if not token:
            break
    return title_to_id

def is_playlist_id_valid(youtube, playlist_id: str) -> bool:
    if not playlist_id:
        return False
    try:
        resp = youtube.playlists().list(part="id", id=playlist_id, maxResults=1).execute()
        return bool(resp.get("items"))
    except Exception:
        return False

def ensure_playlist(youtube, category: str, playlist_map: dict, prefix="YTPG",
                    daily_budget=None, used_units_ref=None, dry_run=False):
    """
    Garantit un playlistId pour 'category' sans duplicats :
      1) Mapping → si ID valide, on réutilise.
      2) Sinon, recherche par titre exact → on réutilise.
      3) Sinon, création (si pas dry_run & budget OK).
    Incrémente used_units_ref['value'] de 50 UNIQUEMENT lors d'une création.
    """
    title = f"{prefix} {category}"

    # 1) Mapping existant valide
    pl_id = playlist_map.get(category)
    if pl_id and is_playlist_id_valid(youtube, pl_id):
        return pl_id

    # 2) Recherche par titre
    try:
        existing = list_user_playlists_by_title(youtube)
        if title in existing:
            pl_id = existing[title]
            playlist_map[category] = pl_id
            return pl_id
    except Exception as e:
        print(f"⚠️ Impossible de lister les playlists existantes: {e}")

    # 3) Création nécessaire
    if dry_run:
        pl_id = f"DRYRUN_{category}"
        playlist_map[category] = pl_id
        return pl_id

    if daily_budget is not None and used_units_ref is not None:
        if used_units_ref["value"] + 50 > daily_budget:
            raise RuntimeError("Budget quotidien atteint avant création de playlist.")

    resp = youtube.playlists().insert(
        part="snippet,status",
        body={
            "snippet": {
                "title": title,
                "description": f"Playlist générée automatiquement pour '{category}'."
            },
            "status": {"privacyStatus": "private"}
        }
    ).execute()
    pl_id = resp["id"]
    playlist_map[category] = pl_id
    if used_units_ref is not None:
        used_units_ref["value"] += 50
    return pl_id

# ---------------- HELPER: contenu existant d’une playlist (skip doublons) ----------------
def get_existing_playlist_video_ids(youtube, playlist_id: str, daily_budget: int, used_units_ref: dict,
                                    per_page_cost: int = 1, max_pages: int = 9999):
    """
    Retourne un set des vidéoIds présents dans la playlist.
    Compte per_page_cost unité(s) par page lue dans used_units_ref['value'].
    Stoppe si dépasserait le budget.
    """
    existing, token, pages = set(), None, 0
    while True:
        if used_units_ref["value"] + per_page_cost > daily_budget:
            print("⏸ Listing interrompu : budget quotidien dépassé si on continue.")
            break
        resp = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=token
        ).execute()
        used_units_ref["value"] += per_page_cost
        for it in resp.get("items", []):
            vid = (it.get("contentDetails") or {}).get("videoId")
            if vid:
                existing.add(vid)
        token = resp.get("nextPageToken")
        pages += 1
        if not token or pages >= max_pages:
            break
    return existing

# ---------------- CHARGEMENT DES TÂCHES (Drive en priorité) ----------------
def load_tasks_from_resume(path_json: str):
    return load_json_from_drive_or_local(path_json)

def load_tasks_from_csv(path_csv: str, include_uncat: bool) -> list:
    # Drive prioritaire
    csv_path = drive_path(path_csv) if (USE_DRIVE and DRIVE_MOUNTED and os.path.exists(drive_path(path_csv))) else \
               (path_csv if os.path.exists(path_csv) else None)
    if not csv_path:
        # tentative d'upload (optionnel)
        try:
            from google.colab import files  # type: ignore
            print("👉 Uploadez votre rapport CSV si vous ne l'avez pas en Drive/local.")
            uploaded = files.upload()
            if uploaded:
                csv_path = next((n for n in uploaded if n.endswith(".csv")), None)
        except Exception:
            pass
    if not csv_path:
        raise FileNotFoundError("CSV introuvable (Drive/local).")

    df = pd.read_csv(csv_path)
    cat_col = 'assigned_category' if 'assigned_category' in df.columns else ('category' if 'category' in df.columns else None)
    if 'video_id' not in df.columns or not cat_col:
        raise ValueError(f"CSV invalide. Colonnes requises: 'video_id' + 'assigned_category'/'category'. Colonnes: {list(df.columns)}")

    if not include_uncat:
        df = df[df[cat_col] != "Non Catégorisé"].copy()

    grouped = df.groupby(cat_col)['video_id'].apply(list).to_dict()
    return [{"category": cat, "videos": vids} for cat, vids in grouped.items() if vids]

# 1) Priorité au JSON de reprise dans Drive
remaining_tasks = None
if file_exists_drive_or_local(RESUME_NAME):
    print(f"♻️ Reprise détectée → {drive_path(RESUME_NAME) if (USE_DRIVE and DRIVE_MOUNTED and os.path.exists(drive_path(RESUME_NAME))) else RESUME_NAME}")
    try:
        remaining_tasks = load_tasks_from_resume(RESUME_NAME)
    except Exception as e:
        print(f"⚠️ Lecture de {RESUME_NAME} échouée : {e}")

# 2) Sinon, reconstruction depuis CSV (Drive > local > upload)
if not remaining_tasks:
    try:
        remaining_tasks = load_tasks_from_csv(REPORT_CSV_NAME, INCLUDE_UNCATEGORIZED)
        print(f"📄 Tâches reconstruites depuis CSV → {REPORT_CSV_NAME}")
    except Exception as e:
        raise SystemExit(f"🛑 Aucune source de tâches trouvée (ni {RESUME_NAME}, ni CSV lisible). Détail: {e}")

# ---------------- STATS DES TÂCHES ----------------
total_playlists = len(remaining_tasks)
total_videos    = sum(len(t["videos"]) for t in remaining_tasks)
estimated_units = total_playlists*50 + total_videos*50
print("\n--- Récap des tâches à traiter ---")
print(f"• Catégories : {total_playlists}")
print(f"• Vidéos     : {total_videos}")
print(f"• Estimation si tout aujourd'hui : {estimated_units} unités (budget {DAILY_QUOTA_BUDGET})")

# ---------------- MAPPING PLAYLISTS (Drive prioritaire) ----------------
playlist_map = load_json_from_drive_or_local(PLAYLIST_MAP_NAME) or {}
if playlist_map:
    print(f"🔗 Mapping playlists chargé ({len(playlist_map)} entrées).")
else:
    print("ℹ️ Pas de mapping existant — repêchage par TITRE et création au besoin.")

# Compteur global d’unités (création + list + insert)
used_units = {"value": 0}

# 1) Préparer/valider toutes les playlists (réutilisation / auto-recreate)
print("\n➡️ Préparation des playlists (réutilisation / recréation si supprimée)…")
for task in tqdm(remaining_tasks, desc="Playlists prêtes"):
    cat = task["category"]
    if not task["videos"]:
        continue
    try:
        pl_id = ensure_playlist(
            youtube=youtube,
            category=cat,
            playlist_map=playlist_map,
            prefix=PLAYLIST_PREFIX,
            daily_budget=DAILY_QUOTA_BUDGET if not DRY_RUN else None,
            used_units_ref=used_units if not DRY_RUN else None,
            dry_run=DRY_RUN
        )
        if str(pl_id).startswith("DRYRUN_"):
            tqdm.write(f"🧪 {cat}: mapping simulé → {pl_id}")
        else:
            tqdm.write(f"✅ {cat}: playlist prête → {pl_id}")
    except RuntimeError as e:
        tqdm.write(f"⏸ {cat}: {e} — création reportée.")
        continue

# Sauvegarder/mettre à jour le mapping dans Drive
save_json_atomic_to_drive(PLAYLIST_MAP_NAME, playlist_map, make_backup=True)
print(f"✅ Mapping playlists sauvegardé dans Drive → {drive_path(PLAYLIST_MAP_NAME) if (USE_DRIVE and DRIVE_MOUNTED) else PLAYLIST_MAP_NAME}")

# 2) Ajout des vidéos (quota-aware, skip doublons, reprise)
print("\n➡️ Ajout des vidéos (quota-aware, multi-jours, sans doublons)…")

moved_count = 0
still_remaining = []

if DRY_RUN:
    print("🧪 DRY_RUN: simulation d’ajout (aucun appel API).")
    for task in remaining_tasks:
        cat, vids = task["category"], task["videos"]
        for _ in vids:
            if used_units["value"] + 50 > DAILY_QUOTA_BUDGET:
                still_remaining.append(task)
                break
            used_units["value"] += 50
            moved_count += 1
    if still_remaining:
        save_json_atomic_to_drive(RESUME_NAME, still_remaining, make_backup=True)
        print(f"⏸ Simulation stoppée. Reste à faire → {drive_path(RESUME_NAME) if (USE_DRIVE and DRIVE_MOUNTED) else RESUME_NAME}")
else:
    for task in tqdm(remaining_tasks, desc="Ajout vidéos"):
        cat, vids = task["category"], task["videos"]
        # Re-sécuriser la playlist (au cas où)
        try:
            pl_id = ensure_playlist(
                youtube=youtube,
                category=cat,
                playlist_map=playlist_map,
                prefix=PLAYLIST_PREFIX,
                daily_budget=DAILY_QUOTA_BUDGET,
                used_units_ref=used_units,  # +50 uniquement si création
                dry_run=False
            )
        except RuntimeError as e:
            tqdm.write(f"⏸ '{cat}': {e} — on reporte toute la catégorie.")
            still_remaining.append(task)
            continue

        # Charger les IDs déjà présents (coûte ~1 unité/page)
        try:
            existing_ids = get_existing_playlist_video_ids(
                youtube=youtube,
                playlist_id=pl_id,
                daily_budget=DAILY_QUOTA_BUDGET,
                used_units_ref=used_units,
                per_page_cost=1
            )
        except Exception as e:
            tqdm.write(f"⚠️ Listing contenu playlist '{cat}' ignoré: {e}")
            existing_ids = set()

        skipped_dupes = 0
        remaining_vids = []

        for vid in vids:
            if vid in existing_ids:   # déjà présent
                skipped_dupes += 1
                continue
            if used_units["value"] + 50 > DAILY_QUOTA_BUDGET:  # stop si budget atteint
                remaining_vids.append(vid)
                continue
            try:
                youtube.playlistItems().insert(
                    part="snippet",
                    body={
                        "snippet": {
                            "playlistId": pl_id,
                            "resourceId": {"kind": "youtube#video", "videoId": vid}
                        }
                    }
                ).execute()
                used_units["value"] += 50
                moved_count += 1
                existing_ids.add(vid)
                time.sleep(0.1)
            except HttpError as e:
                tqdm.write(f"  -> Erreur ajout {vid} dans '{cat}': {e}")
                remaining_vids.append(vid)

        if skipped_dupes:
            tqdm.write(f"🔁 '{cat}': {skipped_dupes} vidéo(s) déjà présentes — ignorées.")

        if remaining_vids:
            still_remaining.append({"category": cat, "videos": remaining_vids})

    # Sauvegarder le mapping (si ensure_playlist a éventuellement créé/relié des IDs)
    save_json_atomic_to_drive(PLAYLIST_MAP_NAME, playlist_map, make_backup=True)

# 3) Sauvegarde de reprise (Drive prioritaire)
if still_remaining:
    save_json_atomic_to_drive(RESUME_NAME, still_remaining, make_backup=True)
    print(f"\n⏸ Stop avant dépassement. {len(still_remaining)} catégories incomplètes.")
    print(f"💾 Reprendre via {drive_path(RESUME_NAME) if (USE_DRIVE and DRIVE_MOUNTED) else RESUME_NAME}")
else:
    # supprimer le resume si plus rien
    if file_exists_drive_or_local(RESUME_NAME):
        try:
            p = drive_path(RESUME_NAME) if (USE_DRIVE and DRIVE_MOUNTED and os.path.exists(drive_path(RESUME_NAME))) else RESUME_NAME
            os.remove(p)
            print("\n🧹 Reprise terminée : fichier de reprise supprimé.")
        except Exception:
            pass
    print("\n✅ Toutes les vidéos planifiées aujourd'hui ont été ajoutées.")

print(f"\nRésumé : {moved_count} vidéos ajoutées aujourd'hui, {used_units['value']} unités consommées.")

# 4) Export (si un CSV source est dispo) — et copie dans Drive si possible
if file_exists_drive_or_local(REPORT_CSV_NAME):
    try:
        # Chemin CSV: priorité Drive
        csv_path = drive_path(REPORT_CSV_NAME) if (USE_DRIVE and DRIVE_MOUNTED and os.path.exists(drive_path(REPORT_CSV_NAME))) else \
                   (REPORT_CSV_NAME if os.path.exists(REPORT_CSV_NAME) else None)

        if csv_path:
            df_rep = pd.read_csv(csv_path)
            cols_pref = ['video_id','video_title','channel']
            cat_col = 'assigned_category' if 'assigned_category' in df_rep.columns else ('category' if 'category' in df_rep.columns else None)
            cols = [c for c in cols_pref if c in df_rep.columns]
            if cat_col: cols.append(cat_col)
            if 'url' in df_rep.columns: cols.append('url')

            # Export Excel répliqué dans Drive
            out_xlsx = drive_path(REPORT_XLSX_NAME) if (USE_DRIVE and DRIVE_MOUNTED) else REPORT_XLSX_NAME
            with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
                df_rep[cols].to_excel(writer, index=False, sheet_name="Rapport Global")
                if cat_col:
                    for cat in sorted(df_rep[cat_col].dropna().unique().tolist()):
                        mask = (df_rep[cat_col] == cat)
                        sheet_name = re.sub(r'[\\/*?:\\[\\]]', '_', str(cat))[:31]
                        df_rep.loc[mask, cols].to_excel(writer, index=False, sheet_name=sheet_name)
            print(f"✅ Rapport Excel → {out_xlsx}")
    except Exception as e:
        print(f"ℹ️ Export ignoré (CSV illisible) : {e}")

print("\n🎉 Étape 7B (Autonome + Drive) terminée.")


=== Étape 7B (Autonome + Drive) — démarrage ===
• DRY_RUN = False
• Budget quotidien (unités) = 9000
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 État persistant → /content/drive/MyDrive/youtube_sorter_state
♻️ Reprise détectée → /content/drive/MyDrive/youtube_sorter_state/remaining_to_add.json

--- Récap des tâches à traiter ---
• Catégories : 5
• Vidéos     : 2337
• Estimation si tout aujourd'hui : 117100 unités (budget 9000)
🔗 Mapping playlists chargé (15 entrées).

➡️ Préparation des playlists (réutilisation / recréation si supprimée)…


Playlists prêtes:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Philosophy & Big Ideas: playlist prête → PLDGCJFyM4KhdKO6lV7i0tRiohj9vHra34
✅ Psychology & Self-Improvement: playlist prête → PLDGCJFyM4KhcnUgiRaG7J8KbY7-XmvFLg
✅ Science & Reality: playlist prête → PLDGCJFyM4Khf3bOIyY_R-89gg-f8SBi9I
✅ Strength, Fitness & Longevity: playlist prête → PLDGCJFyM4Khdc7o_YScuqRfzsfydq0cfr
✅ UFOs, Anomalies & Esoterica: playlist prête → PLDGCJFyM4Kheg5IRfc1FYQWjY_pYedj9T
✅ Mapping playlists sauvegardé dans Drive → /content/drive/MyDrive/youtube_sorter_state/playlist_map.json

➡️ Ajout des vidéos (quota-aware, multi-jours, sans doublons)…


Ajout vidéos:   0%|          | 0/5 [00:00<?, ?it/s]

  -> Erreur ajout Ki3d1e3IGak dans 'Philosophy & Big Ideas': <HttpError 400 when requesting https://youtube.googleapis.com/youtube/v3/playlistItems?part=snippet&alt=json returned "Precondition check failed.". Details: "[{'message': 'Precondition check failed.', 'domain': 'global', 'reason': 'failedPrecondition'}]">
  -> Erreur ajout JKn_DXzsFIQ dans 'Philosophy & Big Ideas': <HttpError 400 when requesting https://youtube.googleapis.com/youtube/v3/playlistItems?part=snippet&alt=json returned "Precondition check failed.". Details: "[{'message': 'Precondition check failed.', 'domain': 'global', 'reason': 'failedPrecondition'}]">
  -> Erreur ajout XVHS9RFjEy4 dans 'Philosophy & Big Ideas': <HttpError 400 when requesting https://youtube.googleapis.com/youtube/v3/playlistItems?part=snippet&alt=json returned "Precondition check failed.". Details: "[{'message': 'Precondition check failed.', 'domain': 'global', 'reason': 'failedPrecondition'}]">
  -> Erreur ajout r3gMvni3vzQ dans 'Philosophy & 

##BAD##

In [ ]:
# ==============================================================================
#   ÉTAPE 7B (AUTONOME) : PLAYLISTS (REUSE/AUTO-RECREATE) + AJOUT QUOTA-AWARE
#                          + SKIP DES DOUBLONS + REPRISE MULTI-JOURS
#   -> Ne dépend que de l'Étape 3 (auth). Aucune référence à classified_df.
#   -> Source des tâches :
#        1) remaining_to_add.json (prioritaire, reprise)
#        2) SAVE_REPORT_CSV (ex: rapport_de_tri.csv) pour reconstruire {cat: [video_ids]}
#        3) invite à uploader si absent
# ==============================================================================

import os, json, re, time
import pandas as pd
from tqdm.auto import tqdm
from googleapiclient.errors import HttpError

# ---------- CONFIG ----------
DAILY_QUOTA_BUDGET = 9000                       # Budget/jour (défaut Google = 10k)
PLAYLIST_PREFIX    = globals().get("PLAYLIST_PREFIX", "YTPG")

# Fichiers d'état
PLAYLIST_MAP_FILE  = "playlist_map.json"        # Mapping {category: playlistId}
RESUME_FILE        = "remaining_to_add.json"    # Backlog {category, videos[]} pour reprise
SAVE_REPORT_CSV    = globals().get("SAVE_REPORT_CSV", "rapport_de_tri.csv")
SAVE_REPORT_XLSX   = globals().get("SAVE_REPORT_XLSX", "rapport_de_tri.xlsx")

# Comportements
INCLUDE_UNCATEGORIZED = False                   # True => inclure "Non Catégorisé"
DRY_RUN = globals().get("DRY_RUN", True)        # sécurité par défaut

# ---------- GUARDS ----------
if 'youtube' not in globals():
    raise SystemExit("🛑 L'API YouTube n'est pas disponible. Exécutez l'Étape 3 (auth) d'abord.")

print("=== Étape 7B (Autonome) — démarrage ===")
print(f"• DRY_RUN = {DRY_RUN}")
print(f"• Budget quotidien (unités) = {DAILY_QUOTA_BUDGET}")

# ---------- HELPERS : anti-duplication & auto-recreate ----------
def list_user_playlists_by_title(youtube):
    """Retourne {title: id} pour toutes vos playlists (mine=True)."""
    title_to_id, token = {}, None
    while True:
        resp = youtube.playlists().list(part="snippet", mine=True, maxResults=50, pageToken=token).execute()
        for it in resp.get("items", []):
            title = (it.get("snippet", {}) or {}).get("title", "").strip()
            plid  = it.get("id")
            if title and plid:
                title_to_id[title] = plid
        token = resp.get("nextPageToken")
        if not token:
            break
    return title_to_id

def is_playlist_id_valid(youtube, playlist_id: str) -> bool:
    if not playlist_id:
        return False
    try:
        resp = youtube.playlists().list(part="id", id=playlist_id, maxResults=1).execute()
        return bool(resp.get("items"))
    except Exception:
        return False

def ensure_playlist(youtube, category: str, playlist_map: dict, prefix="YTPG",
                    daily_budget=None, used_units_ref=None, dry_run=False):
    """
    Garantit un playlistId valide pour 'category' sans duplicats :
      1) Mapping → si ID valide, on réutilise.
      2) Sinon, on cherche par titre exact → on réutilise.
      3) Sinon, on crée (si pas dry_run & budget OK).
    Incrémente used_units_ref['value'] de 50 UNIQUEMENT lors d'une création.
    """
    title = f"{prefix} {category}"

    # 1) Mapping existant valide
    pl_id = playlist_map.get(category)
    if pl_id and is_playlist_id_valid(youtube, pl_id):
        return pl_id

    # 2) Recherche par titre exact
    try:
        existing = list_user_playlists_by_title(youtube)
        if title in existing:
            pl_id = existing[title]
            playlist_map[category] = pl_id
            return pl_id
    except Exception as e:
        print(f"⚠️ Impossible de lister les playlists existantes: {e}")

    # 3) Création si nécessaire
    if dry_run:
        pl_id = f"DRYRUN_{category}"
        playlist_map[category] = pl_id
        return pl_id

    if daily_budget is not None and used_units_ref is not None:
        if used_units_ref["value"] + 50 > daily_budget:
            raise RuntimeError("Budget quotidien atteint avant création de playlist.")

    resp = youtube.playlists().insert(
        part="snippet,status",
        body={
            "snippet": {
                "title": title,
                "description": f"Playlist générée automatiquement pour '{category}'."
            },
            "status": {"privacyStatus": "private"}
        }
    ).execute()
    pl_id = resp["id"]
    playlist_map[category] = pl_id
    if used_units_ref is not None:
        used_units_ref["value"] += 50
    return pl_id

# ---------- HELPER : récupérer les vidéoIds déjà présentes (quota-aware) ----------
def get_existing_playlist_video_ids(youtube, playlist_id: str, daily_budget: int, used_units_ref: dict,
                                    per_page_cost: int = 1, max_pages: int = 9999):
    """
    Retourne un set des vidéoIds présents dans la playlist.
    Compte per_page_cost unité(s) par page lue dans used_units_ref['value'].
    Stoppe si dépasserait le budget.
    """
    existing, token, pages = set(), None, 0
    while True:
        if used_units_ref["value"] + per_page_cost > daily_budget:
            print("⏸ Listing interrompu : le budget quotidien serait dépassé.")
            break
        resp = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=token
        ).execute()
        used_units_ref["value"] += per_page_cost
        for it in resp.get("items", []):
            vid = (it.get("contentDetails") or {}).get("videoId")
            if vid:
                existing.add(vid)
        token = resp.get("nextPageToken")
        pages += 1
        if not token or pages >= max_pages:
            break
    return existing

# ---------- SOURCES DES TÂCHES ----------
remaining_tasks = None  # liste de dicts: {"category": str, "videos": [ids...]}

def load_tasks_from_resume():
    with open(RESUME_FILE, "r", encoding="utf-8") as f:
        return json.load(f)

def load_tasks_from_csv(path: str, include_uncat: bool) -> list:
    df = pd.read_csv(path)
    # Colonnes minimales attendues
    # video_id, assigned_category (ou category)
    cat_col = 'assigned_category' if 'assigned_category' in df.columns else ('category' if 'category' in df.columns else None)
    if 'video_id' not in df.columns or not cat_col:
        raise ValueError(f"CSV invalide. Colonnes requises: 'video_id' et 'assigned_category'/'category'. Colonnes présentes: {list(df.columns)}")

    if not include_uncat:
        df = df[df[cat_col] != "Non Catégorisé"].copy()

    grouped = df.groupby(cat_col)['video_id'].apply(list).to_dict()
    tasks = [{"category": cat, "videos": vids} for cat, vids in grouped.items() if vids]
    return tasks

# 1) Priorité à remaining_to_add.json (reprise)
if os.path.exists(RESUME_FILE):
    print(f"♻️ Reprise trouvée → {RESUME_FILE}")
    try:
        remaining_tasks = load_tasks_from_resume()
    except Exception as e:
        print(f"⚠️ Impossible de lire {RESUME_FILE}: {e}")

# 2) Sinon, tenter depuis le CSV
if remaining_tasks is None:
    if os.path.exists(SAVE_REPORT_CSV):
        print(f"📄 Rapport CSV détecté → {SAVE_REPORT_CSV}")
        try:
            remaining_tasks = load_tasks_from_csv(SAVE_REPORT_CSV, INCLUDE_UNCATEGORIZED)
        except Exception as e:
            print(f"⚠️ Lecture CSV échouée: {e}")
    else:
        # proposer l'upload si Colab
        try:
            from google.colab import files
            print("👉 Uploadez soit remaining_to_add.json (reprise), soit votre rapport CSV (SAVE_REPORT_CSV).")
            uploaded = files.upload()
            if uploaded:
                if RESUME_FILE in uploaded:
                    remaining_tasks = load_tasks_from_resume()
                elif SAVE_REPORT_CSV in uploaded or any(name.endswith(".csv") for name in uploaded):
                    # si l'utilisateur a uploadé un CSV avec un autre nom, prendre le premier CSV
                    csv_name = SAVE_REPORT_CSV if SAVE_REPORT_CSV in uploaded else next(n for n in uploaded if n.endswith(".csv"))
                    remaining_tasks = load_tasks_from_csv(csv_name, INCLUDE_UNCATEGORIZED)
        except Exception:
            pass

# 3) Échec si aucune source
if not remaining_tasks:
    raise SystemExit("🛑 Aucune source de tâches trouvée. Uploadez 'remaining_to_add.json' ou un CSV de rapport.")

# ---------- QUELQUES STATS ----------
total_playlists = len(remaining_tasks)
total_videos    = sum(len(t["videos"]) for t in remaining_tasks)
estimated_units = total_playlists*50 + total_videos*50
print("\n--- Récap des tâches à traiter ---")
print(f"• Catégories à traiter : {total_playlists}")
print(f"• Vidéos à ajouter     : {total_videos}")
print(f"• Estimation unités si tout aujourd'hui : {estimated_units} (budget {DAILY_QUOTA_BUDGET})")

# ---------- MAPPING PLAYLISTS ----------
playlist_map = {}
if os.path.exists(PLAYLIST_MAP_FILE):
    with open(PLAYLIST_MAP_FILE, "r", encoding="utf-8") as f:
        playlist_map = json.load(f)
else:
    print("ℹ️ Pas de mapping local → on réutilisera par TITRE (sinon création).")

# Compteur global d’unités consommées (création + list + insert)
used_units = {"value": 0}

# Préparer / valider toutes les playlists nécessaires (sans doublons)
print("\n➡️ Préparation des playlists (réutilisation / recréation si supprimée)…")
for task in tqdm(remaining_tasks, desc="Playlists ready"):
    cat = task["category"]
    if not task["videos"]:
        continue
    try:
        pl_id = ensure_playlist(
            youtube=youtube,
            category=cat,
            playlist_map=playlist_map,
            prefix=PLAYLIST_PREFIX,
            daily_budget=DAILY_QUOTA_BUDGET if not DRY_RUN else None,
            used_units_ref=used_units if not DRY_RUN else None,
            dry_run=DRY_RUN
        )
        if str(pl_id).startswith("DRYRUN_"):
            tqdm.write(f"🧪 {cat}: mapping simulé → {pl_id}")
        else:
            tqdm.write(f"✅ {cat}: playlist prête → {pl_id}")
    except RuntimeError as e:
        tqdm.write(f"⏸ {cat}: {e} — création reportée.")
        continue

# Sauvegarder le mapping
with open(PLAYLIST_MAP_FILE, "w", encoding="utf-8") as f:
    json.dump(playlist_map, f, indent=2, ensure_ascii=False)
print(f"✅ Mapping playlists → {PLAYLIST_MAP_FILE}")

# ---------- AJOUT DES VIDÉOS (quota-aware, skip doublons, reprise) ----------
print("\n➡️ Ajout des vidéos (quota-aware, multi-jours, sans doublons)…")

moved_count = 0
still_remaining = []

if DRY_RUN:
    print("🧪 DRY_RUN: simulation d’ajout sans appeler l’API.")
    for task in remaining_tasks:
        cat, vids = task["category"], task["videos"]
        for _ in vids:
            if used_units["value"] + 50 > DAILY_QUOTA_BUDGET:
                still_remaining.append(task)
                break
            used_units["value"] += 50
            moved_count += 1
    if still_remaining:
        with open(RESUME_FILE, "w", encoding="utf-8") as f:
            json.dump(still_remaining, f, indent=2, ensure_ascii=False)
        print(f"⏸ Simulation stoppée. Reste à faire → {RESUME_FILE}")
else:
    for task in tqdm(remaining_tasks, desc="Ajout vidéos"):
        cat, vids = task["category"], task["videos"]
        # Re-sécuriser la playlist (au cas où)
        try:
            pl_id = ensure_playlist(
                youtube=youtube,
                category=cat,
                playlist_map=playlist_map,
                prefix=PLAYLIST_PREFIX,
                daily_budget=DAILY_QUOTA_BUDGET,
                used_units_ref=used_units,  # +50 seulement si création
                dry_run=False
            )
        except RuntimeError as e:
            tqdm.write(f"⏸ '{cat}': {e} — on reporte toute la catégorie.")
            still_remaining.append(task)
            continue

        # Charger les IDs déjà présents (coûte ~1 unité/page)
        existing_ids = set()
        try:
            existing_ids = get_existing_playlist_video_ids(
                youtube=youtube,
                playlist_id=pl_id,
                daily_budget=DAILY_QUOTA_BUDGET,
                used_units_ref=used_units,
                per_page_cost=1
            )
        except Exception as e:
            tqdm.write(f"⚠️ Listing contenu playlist '{cat}' ignoré: {e}")

        skipped_dupes = 0
        remaining_vids = []

        for vid in vids:
            # Skip si déjà présent
            if vid in existing_ids:
                skipped_dupes += 1
                continue

            # Stop si budget atteint
            if used_units["value"] + 50 > DAILY_QUOTA_BUDGET:
                remaining_vids.append(vid)
                continue

            try:
                youtube.playlistItems().insert(
                    part="snippet",
                    body={
                        "snippet": {
                            "playlistId": pl_id,
                            "resourceId": {"kind": "youtube#video", "videoId": vid}
                        }
                    }
                ).execute()
                used_units["value"] += 50
                moved_count += 1
                existing_ids.add(vid)  # éviter doublon ultérieur
                time.sleep(0.1)
            except HttpError as e:
                tqdm.write(f"  -> Erreur ajout {vid} dans '{cat}': {e}")
                remaining_vids.append(vid)

        if skipped_dupes:
            tqdm.write(f"🔁 '{cat}': {skipped_dupes} vidéo(s) déjà présentes — ignorées.")

        if remaining_vids:
            still_remaining.append({"category": cat, "videos": remaining_vids})

    # Mapping rafraîchi (si création au passage)
    with open(PLAYLIST_MAP_FILE, "w", encoding="utf-8") as f:
        json.dump(playlist_map, f, indent=2, ensure_ascii=False)

# Sauvegarde de reprise
if still_remaining:
    with open(RESUME_FILE, "w", encoding="utf-8") as f:
        json.dump(still_remaining, f, indent=2, ensure_ascii=False)
    print(f"\n⏸ Stop avant dépassement. {len(still_remaining)} catégories incomplètes.")
    print(f"💾 Reprendre via {RESUME_FILE}")
else:
    if os.path.exists(RESUME_FILE):
        os.remove(RESUME_FILE)
    print("\n✅ Toutes les vidéos planifiées aujourd'hui ont été ajoutées.")

print(f"\nRésumé : {moved_count} vidéos ajoutées aujourd'hui, {used_units['value']} unités consommées.")

# ---------- EXPORT (si un CSV source a été utilisé / dispo) ----------
# Si SAVE_REPORT_CSV existe, on regénère aussi un Excel multi-feuilles pour confort.
if os.path.exists(SAVE_REPORT_CSV):
    try:
        df_rep = pd.read_csv(SAVE_REPORT_CSV)
        # colonnes minimales pour export propre
        cols_pref = ['video_id','video_title','channel']
        cat_col = 'assigned_category' if 'assigned_category' in df_rep.columns else ('category' if 'category' in df_rep.columns else None)
        cols = [c for c in cols_pref if c in df_rep.columns]
        if cat_col: cols.append(cat_col)
        if 'url' in df_rep.columns: cols.append('url')

        # Export CSV (nettoyé)
        out_csv = SAVE_REPORT_CSV  # on réutilise le nom configuré
        df_rep[cols].to_csv(out_csv, index=False, encoding="utf-8-sig")
        print(f"✅ Rapport CSV → {out_csv}")

        # Export Excel si possible
        out_xlsx = SAVE_REPORT_XLSX
        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            df_rep[cols].to_excel(writer, index=False, sheet_name="Rapport Global")
            if cat_col:
                for cat in sorted(df_rep[cat_col].dropna().unique().tolist()):
                    mask = (df_rep[cat_col] == cat)
                    sheet_name = re.sub(r'[\\/*?:\\[\\]]', '_', str(cat))[:31]
                    df_rep.loc[mask, cols].to_excel(writer, index=False, sheet_name=sheet_name)
        print(f"✅ Rapport Excel → {out_xlsx}")
        try:
            from google.colab import files
            files.download(out_csv)
            files.download(out_xlsx)
        except Exception:
            pass
    except Exception as e:
        print(f"ℹ️ Export ignoré (CSV illisible) : {e}")

print("\n🎉 Étape 7B (Autonome) terminée.")


## ÉTAPE 7B : Reprendre un Traitement Interrompu

**N'exécutez cette cellule QUE si votre session précédente a été interrompue** et que vous avez téléchargé un fichier `remaining_to_add.json`.

### Workflow de Reprise :
1.  **Ré-exécutez la cellule d'authentification (ÉTAPE 3)** pour vous connecter à nouveau aux APIs.
2.  Exécutez CETTE cellule.
3.  Vous serez invité à **importer votre fichier `remaining_to_add.json`**.
4.  Le script lira ce fichier et continuera exactement là où il s'était arrêté, en respectant à nouveau votre budget de quota.

In [9]:
# ==============================================================================
#   ÉTAPE 7B : Reprendre un Traitement Interrompu
# ==============================================================================
import json
import os
from google.colab import files
from googleapiclient.errors import HttpError
from tqdm.auto import tqdm
import time

# --- CONFIG ---
DAILY_QUOTA_BUDGET = 9000
RESUME_FILE = "remaining_to_add.json" # Nom du fichier local après upload

# --- 0. Vérification initiale ---
if 'youtube' not in locals():
    raise SystemExit("🛑 ERREUR : L'objet API YouTube n'a pas été trouvé. Veuillez d'abord exécuter la cellule d'authentification (ÉTAPE 3).")
if DRY_RUN:
    raise SystemExit("🛑 ERREUR : Le mode reprise n'est pas compatible avec DRY_RUN=True. Passez DRY_RUN à False dans les paramètres globaux (ÉTAPE 1) et relancez.")

# --- 1. Import du fichier de reprise ---
print(f"👉 Veuillez importer votre fichier de reprise '{RESUME_FILE}'.")
uploaded = files.upload()

if not uploaded or RESUME_FILE not in uploaded:
    raise SystemExit(f"🛑 ERREUR : Le fichier '{RESUME_FILE}' n'a pas été importé.")

print(f"✅ Fichier '{RESUME_FILE}' chargé avec succès.")
with open(RESUME_FILE, 'r', encoding='utf-8') as f:
    tasks_to_process = json.load(f)

# --- 2. Estimation du coût API pour les tâches restantes ---
total_playlists = len(tasks_to_process)
total_videos = sum(len(task['videos']) for task in tasks_to_process)
estimated_units = (total_playlists * 50) + (total_videos * 50)
print(f"\n📊 Estimation du coût API restant : {estimated_units} unités (budget quotidien = {DAILY_QUOTA_BUDGET})")

if estimated_units > DAILY_QUOTA_BUDGET:
    print(f"⚠️ L'estimation dépasse le budget. Le script s'arrêtera avant d'atteindre la limite et créera un nouveau fichier de reprise.")

# --- 3. Reprise de la création des playlists et de l'ajout des vidéos ---
moved_count = 0
used_units = 0
still_remaining_tasks = []

with tqdm(tasks_to_process, desc="Reprise du traitement") as pbar:
    for task in pbar:
        category = task["category"]
        videos_to_add = task["videos"]
        pbar.set_postfix_str(f"Catégorie: {category}")

        # --- A. Créer la playlist (coûte 50 unités) ---
        if used_units + 50 > DAILY_QUOTA_BUDGET:
            pbar.write(f"Quota presque atteint. Report de '{category}' et des tâches restantes.")
            still_remaining_tasks.append(task)
            continue

        try:
            pbar.write(f"📁 Création de la playlist pour '{category}'...")
            playlist_response = youtube.playlists().insert(
                part="snippet,status",
                body={
                    "snippet": {
                        "title": f"{PLAYLIST_PREFIX} {category}",
                        "description": f"Playlist générée automatiquement pour la catégorie '{category}'."
                    },
                    "status": {"privacyStatus": "private"}
                }
            ).execute()
            playlist_id = playlist_response['id']
            used_units += 50
            tqdm.write(f"✅ Playlist '{category}' créée (ID: {playlist_id}). Coût: 50 unités.")
        except HttpError as e:
            tqdm.write(f"🛑 Erreur lors de la création de la playlist '{category}': {e}")
            still_remaining_tasks.append(task)
            continue

        # --- B. Ajouter les vidéos restantes (chacune coûte 50 unités) ---
        remaining_videos_for_this_task = []
        for video_id in videos_to_add:
            if used_units + 50 > DAILY_QUOTA_BUDGET:
                remaining_videos_for_this_task.append(video_id)
                continue

            try:
                youtube.playlistItems().insert(
                    part="snippet",
                    body={
                        "snippet": {
                            "playlistId": playlist_id,
                            "resourceId": {"kind": "youtube#video", "videoId": video_id}
                        }
                    }
                ).execute()
                moved_count += 1
                used_units += 50
                time.sleep(0.1)
            except HttpError as e:
                tqdm.write(f"  -> ⚠️ Erreur lors de l'ajout de {video_id}: {e}")

        if remaining_videos_for_this_task:
            pbar.write(f"Quota atteint. {len(remaining_videos_for_this_task)} vidéos restantes pour '{category}'.")
            still_remaining_tasks.append({
                "category": category,
                "videos": remaining_videos_for_this_task
            })

# --- 4. Sauvegarde de la progression si nécessaire ---
if still_remaining_tasks:
    with open(RESUME_FILE, 'w', encoding='utf-8') as f:
        json.dump(still_remaining_tasks, f, indent=2, ensure_ascii=False)
    print(f"\n⏸ Exécution de nouveau interrompue pour respecter le quota.")
    print(f"💾 Un NOUVEAU fichier '{RESUME_FILE}' a été généré avec les tâches restantes.")
    print(f"🚀 Téléchargez-le et utilisez l'ÉTAPE 7B lors de votre prochaine session pour terminer.")
    files.download(RESUME_FILE)
else:
    if os.path.exists(RESUME_FILE):
        os.remove(RESUME_FILE)
    print("\n✅ Toutes les vidéos restantes ont été déplacées avec succès. Le processus est terminé !")

print(f"\n📊 Résumé de la session : {moved_count} vidéos déplacées, {used_units} unités de quota utilisées.")

👉 Veuillez importer votre fichier de reprise 'remaining_to_add.json'.


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
#   ÉTAPE 8 : VISUALISATION (optionnel, inchangé)
# ==============================================================================
import os
os.system("pip install --quiet seaborn matplotlib")
import matplotlib.pyplot as plt
import seaborn as sns

if 'classified_df' in locals():
    print("--- Démarrage de l'étape 8 : Visualisation et Statistiques ---")
    df_viz = classified_df.copy()
    df_viz['publishedAt'] = pd.to_datetime(df_viz['publishedAt'], errors='coerce')
    sns.set_theme(style="whitegrid")
    # 1) Cat distribution
    plt.figure(figsize=(12, 8))
    category_counts = df_viz['assigned_category'].value_counts()
    sns.barplot(x=category_counts.values, y=category_counts.index)
    plt.title('Nombre de Vidéos par Catégorie'); plt.tight_layout(); plt.show()
    # 2) Top channels
    plt.figure(figsize=(12, 8))
    channel_counts = df_viz['channel'].value_counts().head(15)
    sns.barplot(x=channel_counts.values, y=channel_counts.index)
    plt.title('Top 15 Chaînes'); plt.tight_layout(); plt.show()
    # 3) Trend
    plt.figure(figsize=(15, 6))
    monthly_counts = df_viz.set_index('publishedAt').resample('M').size()
    monthly_counts.plot(kind='line', marker='o', linestyle='-')
    plt.title('Vidéos par Mois de Publication'); plt.tight_layout(); plt.show()
else:
    print("🛑 Le DataFrame `classified_df` n'a pas été trouvé. Exécutez l’étape 5.")


In [ ]:
# ==============================================================================
#   ÉTAPE 9 : STATS & VISUALISATIONS AVANCÉES
# ==============================================================================
# Ajoute : Heatmap jour/heure, Durée moyenne par catégorie (si dispo),
# Treemap Catégorie→Chaîne, Top chaînes par vues cumulées (si dispo),
# Évolution temporelle par catégorie, Wordclouds par catégorie,
# Corrélation Catégorie↔Abonnements (si dispo), Analyse "âge" (publication→ajout)
# ==============================================================================

# -- Install libs nécessaires (silencieux)
import sys, os, json, math, re, time, random
os.system("pip install --quiet squarify wordcloud seaborn matplotlib")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
from wordcloud import WordCloud
from tqdm.auto import tqdm
from datetime import datetime, timezone

sns.set_theme(style="whitegrid")

if 'classified_df' not in locals():
    print("🛑 Le DataFrame `classified_df` n'a pas été trouvé. Exécutez l’Étape 5 avant.")
else:
    print("🔎 Étape 9 : préparation des données…")

    # Copie de travail
    df = classified_df.copy()

    # ------------------------------------------------------------
    # 0) ENRICHISSEMENT OPTIONNEL VIA L’API (si dispo)
    #    - Durée (contentDetails.duration) → duration_seconds
    #    - Vues (statistics.viewCount) → viewCount
    #    - Date d’ajout à la playlist (playlistItems.snippet.publishedAt) → addedAt
    #    - Abonnements (subscriptions.snippet.title) → set des chaînes abonnées
    # ------------------------------------------------------------

    def parse_iso8601_duration_to_seconds(d):
        # Ex: PT1H2M3S
        if not isinstance(d, str) or not d:
            return np.nan
        pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
        m = re.match(pattern, d)
        if not m:
            return np.nan
        h = int(m.group(1) or 0)
        m_ = int(m.group(2) or 0)
        s = int(m.group(3) or 0)
        return h*3600 + m_*60 + s

    # Flags d’enrichissement
    need_duration = 'duration_seconds' not in df.columns
    need_views    = 'viewCount' not in df.columns
    need_added    = 'addedAt' not in df.columns
    need_subs     = 'is_subscribed_channel' not in df.columns

    # A) Récup stats + duration si possible
    if ('youtube' in locals()) and (need_duration or need_views):
        try:
            print("⏬ Enrichissement: récupération contentDetails/statistics…")
            ids = df['video_id'].dropna().unique().tolist()
            batch_size = 50
            durations = {}
            views = {}
            for i in tqdm(range(0, len(ids), batch_size), desc="YouTube videos().list", unit="lot"):
                batch = ids[i:i+batch_size]
                resp = youtube.videos().list(
                    part="contentDetails,statistics",
                    id=",".join(batch),
                    maxResults=50
                ).execute()
                for it in resp.get("items", []):
                    vid = it.get("id")
                    cd = it.get("contentDetails", {})
                    st = it.get("statistics", {})
                    if need_duration and "duration" in cd:
                        durations[vid] = parse_iso8601_duration_to_seconds(cd["duration"])
                    if need_views and "viewCount" in st:
                        try:
                            views[vid] = int(st["viewCount"])
                        except Exception:
                            views[vid] = np.nan
            if need_duration:
                df["duration_seconds"] = df["video_id"].map(durations)
            if need_views:
                df["viewCount"] = df["video_id"].map(views)
        except Exception as e:
            print(f"⚠️ Impossible de récupérer duration/vues : {e}")

    # B) Date d’ajout à la playlist (si SOURCE_MODE='api' et youtube dispo)
    if ('youtube' in locals()) and need_added and 'SOURCE_MODE' in locals() and str(SOURCE_MODE).lower() == "api":
        try:
            print("⏬ Enrichissement: récupération de la date d’ajout à la Watch Later…")
            # Récupère la WL
            wl_id = get_watch_later_playlist_id(youtube)
            added_map = {}
            token = None
            while True:
                req = youtube.playlistItems().list(
                    part="snippet,contentDetails",
                    playlistId=wl_id, maxResults=50, pageToken=token
                )
                resp = req.execute()
                for it in resp.get("items", []):
                    vid = it.get("contentDetails", {}).get("videoId")
                    added = it.get("snippet", {}).get("publishedAt")  # date ajout dans playlist
                    if vid and added:
                        added_map[vid] = added
                token = resp.get("nextPageToken")
                if not token:
                    break
            df["addedAt"] = pd.to_datetime(df["video_id"].map(added_map), errors='coerce')
        except Exception as e:
            print(f"⚠️ Impossible de récupérer la date d’ajout : {e}")

    # C) Abonnements (pour corrélation par catégorie)
    subscribed_channels = set()
    if ('youtube' in locals()) and need_subs:
        try:
            print("⏬ Enrichissement: récupération des abonnements (chaînes)…")
            token = None
            while True:
                resp = youtube.subscriptions().list(
                    part="snippet", mine=True, maxResults=50, pageToken=token
                ).execute()
                for it in resp.get("items", []):
                    ch_title = it.get("snippet", {}).get("title")
                    if ch_title:
                        subscribed_channels.add(ch_title.strip())
                token = resp.get("nextPageToken")
                if not token:
                    break
            df["is_subscribed_channel"] = df["channel"].apply(lambda c: (c or "").strip() in subscribed_channels)
        except Exception as e:
            print(f"⚠️ Impossible de récupérer vos abonnements : {e}")

    # Normalisations de dates
    if 'publishedAt' in df.columns:
        df['publishedAt'] = pd.to_datetime(df['publishedAt'], errors='coerce')

    # ------------------------------------------------------------
    # 1) HEATMAP Jour / Heure de publication
    # ------------------------------------------------------------
    if 'publishedAt' in df.columns and df['publishedAt'].notna().any():
        print("\n📊 Heatmap Jour/Heure de publication")
        tmp = df[['publishedAt']].dropna().copy()
        tmp['weekday'] = tmp['publishedAt'].dt.weekday  # 0=Lundi
        tmp['hour'] = tmp['publishedAt'].dt.hour
        pivot = tmp.pivot_table(index='weekday', columns='hour', values='publishedAt', aggfunc='count').fillna(0)
        # Labels jolis
        idx_map = {0:'Lun',1:'Mar',2:'Mer',3:'Jeu',4:'Ven',5:'Sam',6:'Dim'}
        pivot.index = pivot.index.map(idx_map.get)
        plt.figure(figsize=(14, 5))
        sns.heatmap(pivot, annot=False)
        plt.title("Heatmap : fréquence des publications par jour/heure")
        plt.xlabel("Heure de la journée")
        plt.ylabel("Jour de la semaine")
        plt.tight_layout()
        plt.show()
    else:
        print("\nℹ️ Heatmap jour/heure ignorée (publishedAt indisponible).")

    # ------------------------------------------------------------
    # 2) Durée moyenne par catégorie (si duration_seconds dispo)
    # ------------------------------------------------------------
    if 'duration_seconds' in df.columns and df['duration_seconds'].notna().any():
        print("\n📊 Durée moyenne des vidéos par catégorie (en minutes)")
        g = df.groupby('assigned_category')['duration_seconds'].mean().sort_values(ascending=False) / 60.0
        plt.figure(figsize=(12, 8))
        sns.barplot(x=g.values, y=g.index)
        plt.xlabel("Durée moyenne (minutes)")
        plt.ylabel("Catégorie")
        plt.title("Durée moyenne par catégorie")
        plt.tight_layout()
        plt.show()
    else:
        print("\nℹ️ Durée moyenne par catégorie ignorée (duration_seconds indisponible).")

    # ------------------------------------------------------------
    # 3) Treemap Catégorie → Chaîne (taille = nb vidéos)
    # ------------------------------------------------------------
    print("\n📊 Treemap Catégorie → Chaîne (top paires)")
    # Limiter la taille pour lisibilité : top 15 couples (cat, channel)
    pair_counts = (df.groupby(['assigned_category','channel'])
                     .size().reset_index(name='count')
                     .sort_values('count', ascending=False)
                     .head(15))
    labels = pair_counts.apply(lambda r: f"{r['assigned_category']}\n{r['channel']}\n({r['count']})", axis=1)
    sizes = pair_counts['count'].values
    plt.figure(figsize=(12,8))
    squarify.plot(sizes=sizes, label=labels, alpha=0.8)
    plt.axis('off')
    plt.title("Treemap Catégorie → Chaîne (Top paires)")
    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # 4) Top chaînes par vues cumulées (si viewCount dispo)
    # ------------------------------------------------------------
    if 'viewCount' in df.columns and df['viewCount'].notna().any():
        print("\n📊 Top chaînes par vues cumulées")
        views_by_channel = df.groupby('channel')['viewCount'].sum().sort_values(ascending=False).head(15)
        plt.figure(figsize=(12,8))
        sns.barplot(x=views_by_channel.values, y=views_by_channel.index)
        plt.xlabel("Vues cumulées")
        plt.ylabel("Chaîne")
        plt.title("Top 15 chaînes (vues cumulées)")
        plt.tight_layout()
        plt.show()
    else:
        print("\nℹ️ Top chaînes par vues ignoré (viewCount indisponible).")

    # ------------------------------------------------------------
    # 5) Évolution du nombre de vidéos par catégorie (par mois)
    # ------------------------------------------------------------
    if 'publishedAt' in df.columns and df['publishedAt'].notna().any():
        print("\n📊 Évolution temporelle par catégorie (mensuelle)")
        tmp = df[['assigned_category', 'publishedAt']].dropna().copy()
        tmp['month'] = tmp['publishedAt'].dt.to_period('M').dt.to_timestamp()
        trend = (tmp.groupby(['month','assigned_category'])
                    .size()
                    .reset_index(name='count'))
        plt.figure(figsize=(14,7))
        sns.lineplot(data=trend, x='month', y='count', hue='assigned_category')
        plt.xlabel("Mois de publication")
        plt.ylabel("Nb de vidéos")
        plt.title("Évolution mensuelle par catégorie")
        plt.tight_layout()
        plt.show()
    else:
        print("\nℹ️ Évolution temporelle ignorée (publishedAt indisponible).")

    # ------------------------------------------------------------
    # 6) Wordclouds par catégorie (top 6 catégories pour lisibilité)
    # ------------------------------------------------------------
    print("\n☁️ Wordclouds par catégorie (top 6)")
    try:
        top_cats = df['assigned_category'].value_counts().head(6).index.tolist()
        fig_rows = 2
        fig_cols = 3
        plt.figure(figsize=(16, 9))
        for i, cat in enumerate(top_cats, start=1):
            text = " ".join(df.loc[df['assigned_category']==cat, 'video_title'].astype(str).tolist())
            # Simple nettoyage
            text = re.sub(r'http\S+|[@#]\S+',' ', text)
            wc = WordCloud(width=800, height=400, background_color='white').generate(text if text else "empty")
            ax = plt.subplot(fig_rows, fig_cols, i)
            plt.imshow(wc, interpolation='bilinear'); plt.axis('off'); ax.set_title(cat)
        plt.suptitle("Wordclouds des titres (par catégorie)")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"⚠️ Wordcloud ignoré : {e}")

    # ------------------------------------------------------------
    # 7) Corrélation Catégorie ↔ Abonnements (si dispo)
    # ------------------------------------------------------------
    if 'is_subscribed_channel' in df.columns:
        print("\n📊 % de vidéos provenant de chaînes auxquelles tu es abonné, par catégorie")
        sub_share = (df.groupby('assigned_category')['is_subscribed_channel']
                     .mean()
                     .sort_values(ascending=False) * 100.0)
        plt.figure(figsize=(12,8))
        sns.barplot(x=sub_share.values, y=sub_share.index)
        plt.xlabel("% de vidéos (chaînes abonnées)")
        plt.ylabel("Catégorie")
        plt.title("Part de chaînes abonnées par catégorie")
        plt.tight_layout()
        plt.show()
    else:
        print("\nℹ️ Corrélation abonnements ignorée (abonnements indisponibles).")

    # ------------------------------------------------------------
    # 8) Analyse “Âge” des vidéos (temps entre publication et ajout)
    # ------------------------------------------------------------
    if 'addedAt' in df.columns and df['addedAt'].notna().any() and 'publishedAt' in df.columns:
        print("\n📊 Analyse de l'âge des vidéos (publication → ajout)")
        tmp = df[['publishedAt','addedAt']].dropna().copy()
        # delta en jours
        tmp['age_days'] = (tmp['addedAt'] - tmp['publishedAt']).dt.total_seconds() / 86400.0
        tmp = tmp[np.isfinite(tmp['age_days'])]
        plt.figure(figsize=(12,6))
        sns.histplot(tmp['age_days'], bins=40, kde=True)
        plt.xlabel("Jours écoulés entre publication et ajout à la playlist")
        plt.title("Distribution de l'âge des vidéos au moment de l'ajout")
        plt.tight_layout()
        plt.show()

        # Par catégorie (top 10)
        if 'assigned_category' in df.columns:
            tmp2 = df.dropna(subset=['publishedAt','addedAt']).copy()
            tmp2['age_days'] = (tmp2['addedAt'] - tmp2['publishedAt']).dt.total_seconds() / 86400.0
            cat_age = tmp2.groupby('assigned_category')['age_days'].median().sort_values().head(10)
            plt.figure(figsize=(12,6))
            sns.barplot(x=cat_age.values, y=cat_age.index)
            plt.xlabel("Médiane âge à l'ajout (jours)")
            plt.title("Âge médian à l'ajout par catégorie (Top 10)")
            plt.tight_layout()
            plt.show()
    else:
        print("\nℹ️ Analyse 'âge' ignorée (addedAt indisponible).")

    print("\n✅ Étape 9 terminée.")
